In [1]:
import wikipediaapi

In [2]:
wiki = wikipediaapi.Wikipedia(
    language="en",
    user_agent="KnowledgeAugmentedVQA/0.1 (contact: maazmustafa2692@gmail.com)"
)

In [3]:
page = wiki.page("Lion")
print(page.exists())

True


In [4]:
print(page.title)

Lion


In [6]:
wikipedia_text = page.text

print(wikipedia_text[:4000])

The lion (Panthera leo) is a large cat of the genus Panthera, currently ranging only in Sub-Saharan Africa and India. It has a muscular, broad-chested body; a short, rounded head; round ears; and a dark, hairy tuft at the tip of its tail. It is sexually dimorphic; adult male lions are larger than females and have a prominent mane that extends from the head to the shoulders and chest.
The lion inhabits grasslands, savannahs, and shrublands. It is an apex and keystone predator, preying mostly on medium-sized and large hooved mammals. It is usually more diurnal than other wild cats, but when persecuted, it adapts to being active at night and at twilight. It is a social species, forming groups called prides. A lion pride consists of related females and cubs, and a few or one adult male who is unrelated to the females. Groups of female lions usually hunt together. Adult males often compete to keep or gain their membership in the pride.
During the Neolithic period, the lion ranged throughout

In [7]:
def chunk_text(text, chunk_size=120, overlap=30):
    words = text.split()

    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [8]:
chunks = chunk_text(wikipedia_text)

print("Number of chunks:", len(chunks))

Number of chunks: 95


In [10]:
print(chunks[0])
print(chunks[3])

The lion (Panthera leo) is a large cat of the genus Panthera, currently ranging only in Sub-Saharan Africa and India. It has a muscular, broad-chested body; a short, rounded head; round ears; and a dark, hairy tuft at the tip of its tail. It is sexually dimorphic; adult male lions are larger than females and have a prominent mane that extends from the head to the shoulders and chest. The lion inhabits grasslands, savannahs, and shrublands. It is an apex and keystone predator, preying mostly on medium-sized and large hooved mammals. It is usually more diurnal than other wild cats, but when persecuted, it adapts to being active at night and at twilight. It is a social species, forming groups
films. It is one of the most widely recognised animal symbols in human culture. Lions have been kept in menageries since the time of the Roman Empire and have been a key species sought for exhibition in zoological gardens across the world since the late 18th century. Cultural depictions of lions have

In [11]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
chunk_embeddings = embedding_model.encode(
    chunks,
    normalize_embeddings=True
)

In [13]:
print(chunk_embeddings.shape)

(95, 384)


In [14]:
import faiss
import numpy as np

In [15]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    np.array(chunk_embeddings).astype("float32")
)

print("Chunks stored in FAISS:", index.ntotal)

Chunks stored in FAISS: 95


In [16]:
question = "What do lions eat?"
question_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
)

In [17]:
scores, indices = index.search(
    np.array(question_embedding).astype("float32"),
    k=3
)

In [18]:
for rank, (idx, score) in enumerate(
    zip(indices[0], scores[0]),
    start=1
):
    print(f"\n--- Result {rank} ---")
    print(f"Score: {score:.4f}")
    print(chunks[idx])


--- Result 1 ---
Score: 0.7007
the others and mates more frequently. Hunting and diet The lion is a generalist hypercarnivore and is considered to be both an apex and keystone predator due to its wide prey spectrum. Its prey consists mainly of medium-sized to large hooved mammals, particularly blue wildebeest, plains zebra, African buffalo, gemsbok and giraffe. It also frequently takes common warthog despite it being much smaller. In India, chital and sambar deer are the most common wild prey, while livestock contributes significantly to lion kills outside protected areas. It usually avoids adult elephants, rhinoceros and hippopotamus and small prey like dik-dik, hyraxes, hares and monkeys, and seldom consumes other predators. Young lions first display stalking behaviour at around three months of age, although

--- Result 2 ---
Score: 0.6712
Cubs suffer most when food is scarce, but otherwise, all pride members eat their fill, including old and crippled lions, which can live on leftov